In [15]:

import pandas as pd
from itertools import product

# Read the dataframe
df = pd.read_csv("/Users/surya/Desktop/riverline_takehome/dev/conversation_bundle_flat.csv")

# Helper: for each row, determine which annotation columns are notnull
def annotation_presence(row):
    return tuple(pd.notnull([row["an1"], row["an2"], row["an3"]]))

# Map: combination name -> conversation IDs with that combination
combination_ids = {}

# All combinations: (an1, an2, an3) where each is True (present) or False (missing)
combinations = list(product([False, True], repeat=3))
labels = ['an1', 'an2', 'an3']

for comb in combinations:
    # Build description for print/readability
    if any(comb):
        comb_desc = "only " + ", ".join(l for l, p in zip(labels, comb) if p) if sum(comb) == 1 else " & ".join(l for l, p in zip(labels, comb) if p)
    else:
        comb_desc = "none"
    mask = (pd.notnull(df['an1']) if comb[0] else df['an1'].isnull()) & \
           (pd.notnull(df['an2']) if comb[1] else df['an2'].isnull()) & \
           (pd.notnull(df['an3']) if comb[2] else df['an3'].isnull())
    ids = df.loc[mask, "conversation_id"].unique().tolist()
    combination_ids[comb_desc] = ids

# Print results
for k, v in combination_ids.items():
    print(f"Conversation IDs with {k}: {len(v)}")


Conversation IDs with none: 300
Conversation IDs with only an3: 100
Conversation IDs with only an2: 100
Conversation IDs with an2 & an3: 0
Conversation IDs with only an1: 100
Conversation IDs with an1 & an3: 0
Conversation IDs with an1 & an2: 0
Conversation IDs with an1 & an2 & an3: 100


In [16]:
combination_ids.keys()

dict_keys(['none', 'only an3', 'only an2', 'an2 & an3', 'only an1', 'an1 & an3', 'an1 & an2', 'an1 & an2 & an3'])

In [17]:
# Get 2 conversation IDs from each combination in combination_ids;
two_per_combination = {k: v[:2] for k, v in combination_ids.items()}
two_per_combination 

for k,v in combination_ids.items():
    if len(v) > 2:
        print(f'{k}: {v[:2]}')

none: ['00ad209e-5236-c672-860f-53bee3b366cd', '018ddf34-d2f6-ae46-76fa-33c67ae24235']
only an3: ['003d61b9-9929-0d01-78c0-72d65befe630', '007fd69e-f588-87d7-edef-6847b0ff176e']
only an2: ['022f42f7-2654-206e-c2e5-100b86c215ee', '03f02884-3f00-12b6-2300-93e089e3fec6']
only an1: ['0398595b-e02f-56a1-73e9-3788e6aba235', '04fe76b0-23de-3a1b-d545-915eec901fbc']
an1 & an2 & an3: ['00ab8e48-5dd9-3bb5-c054-216ab58c9fda', '0116e0f9-0373-5d3a-149d-7b18d3c3c579']


In [18]:

combination_ids['an1 & an2 & an3'][:10]

['00ab8e48-5dd9-3bb5-c054-216ab58c9fda',
 '0116e0f9-0373-5d3a-149d-7b18d3c3c579',
 '01d2bc92-a9d8-1c8c-3a16-3bc390f1de16',
 '0261f428-dcd8-21c5-227c-f8d0502cc8d2',
 '032101dc-a78c-9aa9-c8e2-5f92b42f1dfa',
 '051670b7-b772-dc86-433a-099a40b8ab18',
 '05739a43-4de4-af02-6b9c-81092e72bcff',
 '081f905f-2867-2616-d97a-37ee7ebcf7db',
 '0d279dd8-65e4-98e1-549d-f47998bba681',
 '0fdf1e6b-da43-39e8-de9e-5e1e45207173']

In [19]:
import pandas as pd

# Read the dataframe
df = pd.read_csv("/Users/surya/Desktop/riverline_takehome/dev/conversation_bundle_flat.csv")

# Attempt to safely extract the language from the metadata column;
# This handles cases where metadata is a string representation of a dict (e.g. JSON).
import ast

def extract_language(metadata):
    if pd.isnull(metadata):
        return None
    # Try parsing metadata as dict if it's a string; handle errors gracefully
    if isinstance(metadata, dict):
        meta = metadata
    else:
        try:
            meta = ast.literal_eval(str(metadata))
        except Exception:
            return None
    return meta.get("language")

df["language"] = df["metadata"].apply(extract_language)

# Group conversation IDs by language
language_groups = df.groupby("language")["conversation_id"].apply(list).to_dict()

for lang, conv_ids in language_groups.items():
    print(f"Language={lang!r}, Conversation IDs count: {len(conv_ids)}")

Language='english', Conversation IDs count: 250
Language='hindi', Conversation IDs count: 250
Language='hinglish', Conversation IDs count: 200


In [20]:
for lang, conv_ids in language_groups.items():
    sample_ids = conv_ids[:2]
    print(f"Language={lang!r}, 2 sample conversation IDs: {sample_ids}")

Language='english', 2 sample conversation IDs: ['02081e55-d651-39f8-dc14-17e3cbb3cb11', '0261f428-dcd8-21c5-227c-f8d0502cc8d2']
Language='hindi', 2 sample conversation IDs: ['00ab8e48-5dd9-3bb5-c054-216ab58c9fda', '00ad209e-5236-c672-860f-53bee3b366cd']
Language='hinglish', 2 sample conversation IDs: ['003d61b9-9929-0d01-78c0-72d65befe630', '007fd69e-f588-87d7-edef-6847b0ff176e']


In [21]:
import sys
sys.path.append("..")

from visualise_conv import show_conversation

an3_samples= ['003d61b9-9929-0d01-78c0-72d65befe630', '007fd69e-f588-87d7-edef-6847b0ff176e']
an2_samples= ['022f42f7-2654-206e-c2e5-100b86c215ee', '03f02884-3f00-12b6-2300-93e089e3fec6']
an1_samples= ['0398595b-e02f-56a1-73e9-3788e6aba235', '04fe76b0-23de-3a1b-d545-915eec901fbc']
an123_samples=  ['00ab8e48-5dd9-3bb5-c054-216ab58c9fda',
 '0116e0f9-0373-5d3a-149d-7b18d3c3c579',
 '01d2bc92-a9d8-1c8c-3a16-3bc390f1de16',
 '0261f428-dcd8-21c5-227c-f8d0502cc8d2',
 '032101dc-a78c-9aa9-c8e2-5f92b42f1dfa',
 '051670b7-b772-dc86-433a-099a40b8ab18',
 '05739a43-4de4-af02-6b9c-81092e72bcff',
 '081f905f-2867-2616-d97a-37ee7ebcf7db',
 '0d279dd8-65e4-98e1-549d-f47998bba681',
 '0fdf1e6b-da43-39e8-de9e-5e1e45207173']

sample = an123_samples[7]
# Example: show a conversation (replace conversation_id as needed)
show_conversation(sample)

language,english
zone,north
dpd,40
pos,35000
tos,40250
settlement_offered,31500
total_turns,10
payment_received,True
days_to_payment,19
payment_amount,8855
expected_amount,10000


In [25]:
res

{'quality_score': 0.17999999999999994,
 'risk_score': 0.505,
 'violations': [{'turn': -1,
   'rule': 'AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS',
   'severity': 0.8,
   'explanation': 'settlement_offered=234000 not within [POS=250000, TOS=287500]; settlement amount must lie within the allowed band (A3).'},
  {'turn': 6,
   'rule': 'QLT_REPETITIVE_RESPONSE_LOOP',
   'severity': 0.35,
   'explanation': 'Bot repeated identical message at turn 6 (previous identical turn: 5).'},
  {'turn': 10,
   'rule': 'QLT_REPETITIVE_RESPONSE_LOOP',
   'severity': 0.35,
   'explanation': 'Bot repeated identical message at turn 10 (previous identical turn: 9).'},
  {'turn': 14,
   'rule': 'QLT_REPETITIVE_RESPONSE_LOOP',
   'severity': 0.35,
   'explanation': 'Bot repeated identical message at turn 14 (previous identical turn: 13).'}]}

In [22]:
# --- AgentEvaluator: run on all production logs, build per-conversation table ---
import ast
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

DEV = Path(".").resolve()
REPO = DEV.parent
PS = REPO / "problem_statement"
sys.path.insert(0, str(PS))

from eval_takehome import AgentEvaluator, RULES  # noqa: E402

LOG_PATH = PS / "data" / "production_logs.jsonl"
FLAT_CSV = DEV / "conversation_bundle_flat.csv"


def read_jsonl(path: Path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def _language_from_metadata(metadata) -> str | None:
    if pd.isna(metadata):
        return None
    if isinstance(metadata, dict):
        return metadata.get("language")
    try:
        return ast.literal_eval(str(metadata)).get("language")
    except Exception:
        return None


evaluator = AgentEvaluator()
rows: list[dict] = []
viol_rows: list[dict] = []

for conv in read_jsonl(LOG_PATH):
    cid = conv["conversation_id"]
    res = evaluator.evaluate(conv)
    rule_counts: dict[str, int] = {}
    for v in res["violations"]:
        r = v["rule"]
        rule_counts[r] = rule_counts.get(r, 0) + 1
        viol_rows.append({"conversation_id": cid, **v})
    row = {
        "conversation_id": cid,
        "quality_score": res["quality_score"],
        "risk_score": res["risk_score"],
        "n_violations": len(res["violations"]),
    }
    for rule_id in RULES:
        row[f"n_{rule_id}"] = rule_counts.get(rule_id, 0)
    rows.append(row)

eval_df = pd.DataFrame(rows)
viol_df = pd.DataFrame(viol_rows) if viol_rows else pd.DataFrame()

# Language from flat bundle (same CSV as earlier cells)
_flat = pd.read_csv(FLAT_CSV, usecols=["conversation_id", "metadata"])
_flat["language"] = _flat["metadata"].apply(_language_from_metadata)
eval_df = eval_df.merge(_flat[["conversation_id", "language"]], on="conversation_id", how="left")

print(f"Evaluated {len(eval_df)} conversations from {LOG_PATH}")
print("\nScore summary:")
print(eval_df[["quality_score", "risk_score", "n_violations"]].describe().T)
print("\nViolations by rule (total event count):")
_rule_totals = {rid: int(eval_df[f"n_{rid}"].sum()) for rid in RULES}
for rid, cnt in sorted(_rule_totals.items(), key=lambda x: -x[1]):
    if cnt:
        print(f"  {rid}: {cnt}")

display(eval_df.sort_values("risk_score", ascending=False).head(15))
eval_df.head()


Evaluated 700 conversations from /Users/surya/Desktop/riverline_takehome/problem_statement/data/production_logs.jsonl

Score summary:
               count      mean       std   min   25%   50%    75%   max
quality_score  700.0  0.407114  0.215142  0.15  0.15  0.49  0.600   1.0
risk_score     700.0  0.537264  0.261300  0.00  0.40  0.40  0.645   1.0
n_violations   700.0  2.952857  2.552298  0.00  1.00  2.00  4.000  11.0

Violations by rule (total event count):
  QLT_REPETITIVE_RESPONSE_LOOP: 956
  AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS: 460
  INV_EXIT_STATE_NOT_FINAL: 441
  TR_INVALID_STATE_TRANSITION: 169
  TR_BACKWARD_TRANSITION_NOT_ALLOWED: 41


,conversation_id,quality_score,risk_score,n_violations,n_TR_INVALID_STATE_TRANSITION,n_TR_BACKWARD_TRANSITION_NOT_ALLOWED,n_TR_BACKWARD_EXCEPTION_MISUSED,n_INV_EXIT_STATE_NOT_FINAL,n_ACT_REQUEST_SETTLEMENT_AMOUNT_INVALID_CONTEXT,n_ACT_CONFIRM_PAYMENT_INVALID_CONTEXT,n_AMT_POS_GREATER_THAN_TOS,n_AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS,n_QLT_REPETITIVE_RESPONSE_LOOP,language
119,0261f428-dcd8-21c5-227c-f8d0502cc8d2,0.15,1.0,3,1,0,0,2,0,0,0,0,0,english
141,bad44e7e-5ffc-6dd3-2f66-bc55b2cff7f6,0.15,1.0,7,2,0,0,5,0,0,0,0,0,english
131,4fe9f8fc-7adc-ebec-1fe1-1a43f8307243,0.15,1.0,7,2,0,0,5,0,0,0,0,0,english
132,f830b6cb-d001-af00-ad4c-bb0d568921e8,0.15,1.0,3,1,0,0,2,0,0,0,0,0,english
133,1040a60b-cd36-1cc4-a43d-bc557e8cc102,0.15,1.0,4,1,0,0,3,0,0,0,0,0,english
134,800d98e4-ab54-4fd7-8dec-cc9dd7911ab5,0.15,1.0,3,1,0,0,2,0,0,0,0,0,english
135,cd5dad45-eed4-31c5-6aa7-a09c7c192fd3,0.15,1.0,3,1,0,0,2,0,0,0,0,0,english
136,b225a421-85f2-93da-c2bf-e8da5a3238fc,0.15,1.0,6,2,0,0,4,0,0,0,0,0,english
137,5d1f6d7a-6f5d-8b19-77d9-c7edcd55b5bb,0.15,1.0,3,1,0,0,2,0,0,0,0,0,english
138,84c70609-895d-a8dd-e013-61c9bffc93cd,0.15,1.0,4,1,0,0,3,0,0,0,0,0,english


,conversation_id,quality_score,risk_score,n_violations,n_TR_INVALID_STATE_TRANSITION,n_TR_BACKWARD_TRANSITION_NOT_ALLOWED,n_TR_BACKWARD_EXCEPTION_MISUSED,n_INV_EXIT_STATE_NOT_FINAL,n_ACT_REQUEST_SETTLEMENT_AMOUNT_INVALID_CONTEXT,n_ACT_CONFIRM_PAYMENT_INVALID_CONTEXT,n_AMT_POS_GREATER_THAN_TOS,n_AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS,n_QLT_REPETITIVE_RESPONSE_LOOP,language
0,192f029c-2626-7e25-7fee-3fff275530b7,0.60,0.40,1,0,0,0,0,0,0,0,1,0,english
1,5e21f706-117e-4ef1-7e9b-7c3dc135123c,0.60,0.40,1,0,0,0,0,0,0,0,1,0,english
2,66263e7c-a4b2-c28a-7093-285797711127,0.60,0.40,1,0,0,0,0,0,0,0,1,0,english
3,eafa2fdc-e973-3ef8-7af9-09c6a59012a4,0.60,0.40,1,0,0,0,0,0,0,0,1,0,english
4,18318619-8cf2-3e5d-1e1a-62d091808f9c,0.72,0.07,2,0,0,0,0,0,0,0,0,2,english


In [24]:
# --- Broader results: outcomes × evaluator, language, rule mix ---
# Run the evaluation cell above first (defines eval_df, viol_df, read_jsonl, PS).
from IPython.display import display

OUTCOMES_PATH = PS / "data" / "outcomes.jsonl"

outcomes_df = pd.DataFrame(list(read_jsonl(OUTCOMES_PATH)))
merged = eval_df.merge(outcomes_df, on="conversation_id", how="left", suffixes=("", "_outcome"))

# Normalize bool columns for grouping (JSON may be bool or missing)
for c in ["borrower_complained", "regulatory_flag", "payment_received"]:
    if c in merged.columns:
        merged[c] = merged[c].fillna(False).astype(bool)

print("Mean scores by outcome flags")
display(
    merged.groupby("borrower_complained", dropna=False)[["quality_score", "risk_score", "n_violations"]]
    .mean()
    .round(3)
)
display(
    merged.groupby("regulatory_flag", dropna=False)[["quality_score", "risk_score", "n_violations"]]
    .mean()
    .round(3)
)
display(
    merged.groupby("payment_received", dropna=False)[["quality_score", "risk_score", "n_violations"]]
    .mean()
    .round(3)
)

print("\nBy language (from bundle metadata)")
_lang = (
    merged.groupby("language", dropna=False)[["quality_score", "risk_score", "n_violations"]]
    .agg(["mean", "median", "count"])
    .round(3)
)
display(_lang)

# Risk quartiles vs complaint / regulatory rate
# Fix ValueError: Bin labels must be one fewer than the number of bin edges (can happen if there are duplicate edges after `qcut`)
# Instead of fixed labels, let qcut assign default labels and then rename if desired
try:
    merged["risk_quartile"] = pd.qcut(merged["risk_score"], q=4, duplicates="drop")
except ValueError as e:
    print(f"qcut failed: {e}")
    # Fallback: assign everything to a single quartile, or skip the quartile grouping
    merged["risk_quartile"] = "all"
else:
    # Optionally assign names to quartiles if there are exactly 4 bins
    n_quart = merged["risk_quartile"].nunique(dropna=True)
    if n_quart == 4:
        merged["risk_quartile"] = pd.qcut(merged["risk_score"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"], duplicates="drop")
    # Otherwise, keep the Interval labels

print("\nComplaint / regulatory rate by risk quartile")
_q = merged.groupby("risk_quartile", observed=True).agg(
    n=("conversation_id", "count"),
    complained_rate=("borrower_complained", "mean"),
    regulatory_rate=("regulatory_flag", "mean"),
    mean_quality=("quality_score", "mean"),
    mean_risk=("risk_score", "mean"),
)
display(_q.round(3))

# Any high-severity structural flags
merged["any_INV"] = (merged["n_INV_EXIT_STATE_NOT_FINAL"] > 0).astype(int)
merged["any_TR_invalid"] = (merged["n_TR_INVALID_STATE_TRANSITION"] > 0).astype(int)
print("\nComplaint rate when INV_EXIT fires vs not")
display(
    merged.groupby("any_INV")["borrower_complained"].agg(["mean", "count"]).round(3)
)

print("\nPer-rule violation row counts (exploded); use for spot-checks")
if len(viol_df):
    display(viol_df["rule"].value_counts().to_frame("n_rows"))
    display(viol_df.groupby("rule")["severity"].describe().round(3).T)


Mean scores by outcome flags


,quality_score,risk_score,n_violations
borrower_complained,,,
False,0.411,0.530,2.920
True,0.374,0.598,3.218


,quality_score,risk_score,n_violations
regulatory_flag,,,
False,0.406,0.537,2.953
True,0.428,0.546,2.955


,quality_score,risk_score,n_violations
payment_received,,,
False,0.337,0.605,3.693
True,0.564,0.385,1.284



By language (from bundle metadata)


...


Complaint rate when INV_EXIT fires vs not


,mean,count
any_INV,,
0,0.092,555
1,0.186,145



Per-rule violation row counts (exploded); use for spot-checks


,n_rows
rule,
QLT_REPETITIVE_RESPONSE_LOOP,956
AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS,460
INV_EXIT_STATE_NOT_FINAL,441
TR_INVALID_STATE_TRANSITION,169
TR_BACKWARD_TRANSITION_NOT_ALLOWED,41


rule,AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS,INV_EXIT_STATE_NOT_FINAL,QLT_REPETITIVE_RESPONSE_LOOP,TR_BACKWARD_TRANSITION_NOT_ALLOWED,TR_INVALID_STATE_TRANSITION
count,460.0,441.0,956.00,41.00,169.0
mean,0.8,1.0,0.35,0.85,0.8
std,0.0,0.0,0.00,0.00,0.0
min,0.8,1.0,0.35,0.85,0.8
25%,0.8,1.0,0.35,0.85,0.8
50%,0.8,1.0,0.35,0.85,0.8
75%,0.8,1.0,0.35,0.85,0.8
max,0.8,1.0,0.35,0.85,0.8
